# 10 — Which model makes the most money?

The goal is not to be un-arbitrageable, it is to maximise profit. Profit = house edge times
volume, and volume depends on how competitive the odds are. We compare **every model we built**
under **elastic demand** (`demand_responsive_pool`: tighter odds draw more volume) with the two
expected-value-gated arbitrageurs (predictive and regime-aware) betting inside the pool, and read
off the **net house PnL**.

Models compared:
- `momentum_lookup` — original 1D momentum lookup, fixed split.
- `momentum_lookup_rolling` — the same, rolling window.
- `volatility_regime_momentum` — momentum calibrated per volatility regime.
- `momentum_logistic_rolling` — displays a multi-feature logistic estimate of P(up).
- `hidden_symmetric_margin` — model 3: hides direction, charges a symmetric information margin.
- `guarded_volatility_regime_momentum` — defence in depth: regime display plus information margin.

All vig-only models quote the same margin, so they draw the same uninformed volume; they differ
only in how much they leak to the arbitrageurs. Model 3 and the guarded model charge extra margin,
so they trade less uninformed volume. Demand elasticity is 8.0 and total volume is calibrated to
~200,000 at the base vig (see the caveat at the end).

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snapmarket.data import load_oracle_prices
from snapmarket.features import build_features
from snapmarket.parameters import SharedParameters
from snapmarket.models import build_model
from snapmarket.experiments import common_evaluation_start
from snapmarket.signals import walk_forward_logistic_probability, regime_conditional_probability
from snapmarket.strategies import demand_responsive_pool, predictive_bettor, regime_aware_bettor
from snapmarket.engine import simulate

shared_parameters = SharedParameters()
features = build_features(load_oracle_prices(), shared_parameters)

model_names = [
    "momentum_lookup", "momentum_lookup_rolling", "volatility_regime_momentum",
    "momentum_logistic_rolling", "hidden_symmetric_margin", "guarded_volatility_regime_momentum",
]
models = {name: build_model(name, features, shared_parameters) for name in model_names}

evaluation_start = common_evaluation_start(models.values())
logistic_probability = walk_forward_logistic_probability(features, shared_parameters)
regime_probability = regime_conditional_probability(features, shared_parameters)
print(f"common evaluation start ~{evaluation_start / 86_400:.0f} days in")

In [ ]:
elasticity = 8.0
reference_margin = 0.125
target_volume = 200_000
window_length = 300_000

# Calibrate the demand level on a vig-only model so its uninformed volume is ~target_volume.
reference = simulate(models["volatility_regime_momentum"], features,
                     {"pool": demand_responsive_pool(1.0, elasticity, reference_margin)},
                     evaluation_start, window_length, seed=7)
base_stake = target_volume / reference.per_bettor["pool"].stake
print(f"calibrated base stake = {base_stake:.4f}")

In [ ]:
rows = []
for name, model in models.items():
    flow = {
        "pool": demand_responsive_pool(base_stake, elasticity, reference_margin),
        "predictive": predictive_bettor(logistic_probability, base_stake=base_stake),
        "regime_aware": regime_aware_bettor(regime_probability, base_stake=base_stake),
    }
    result = simulate(model, features, flow, evaluation_start, window_length, seed=7)
    attacker_pnl = result.per_bettor["predictive"].pnl + result.per_bettor["regime_aware"].pnl
    rows.append({
        "model": name,
        "uninformed_volume": result.per_bettor["pool"].stake,
        "attacker_volume": result.per_bettor["predictive"].stake + result.per_bettor["regime_aware"].stake,
        "attacker_profit_vs_house": attacker_pnl,
        "net_house_pnl": result.house_pnl,
        "net_house_edge": result.house_edge,
    })

comparison = pd.DataFrame(rows).set_index("model").sort_values("net_house_pnl", ascending=False)
print(comparison.round(2).to_string())
best = comparison["net_house_pnl"].idxmax()
print(f"\nMost profitable model: {best}  (net house PnL {comparison.loc[best, 'net_house_pnl']:,.0f})")

In [ ]:
order = comparison.sort_values("net_house_pnl")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

order["net_house_pnl"].plot(kind="barh", ax=axes[0], color="#2563eb")
axes[0].set_title("Net house PnL by model (elastic demand + arbitrageurs)")
axes[0].set_xlabel("net house PnL")

axes[1].barh(order.index, order["uninformed_volume"], color="#9ca3af", label="uninformed pool volume")
axes[1].barh(order.index, order["attacker_volume"], left=order["uninformed_volume"],
             color="#f59e0b", label="attacker volume")
axes[1].set_title("Volume by source")
axes[1].set_xlabel("USDT"); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Recorded results

Window of 300,000 seconds from the common out-of-sample start; demand elasticity 8.0; uninformed
volume calibrated to ~200,000 at the base vig. `attacker_profit_vs_house` is positive when the
arbitrageurs win (the house loses that amount).

| model | uninformed volume | attacker profit vs house | net house PnL |
|---|---|---|---|
| volatility_regime_momentum | 199,934 | +725 | **25,408** |
| momentum_logistic_rolling | 199,926 | +448 | 25,270 |
| guarded_volatility_regime_momentum | 196,404 | +1,077 | 24,902 |
| momentum_lookup | 199,934 | +1,797 | 24,191 |
| momentum_lookup_rolling | 199,934 | +1,888 | 24,096 |
| hidden_symmetric_margin (model 3) | 188,614 | +3,455 | 21,151 |

**Read.**
- The most profitable model is **`volatility_regime_momentum`** (~25,400), with `momentum_logistic_rolling` essentially tied (~25,300). They price the direction well, so they keep full uninformed volume and leak little to the arbitrageurs.
- **Model 3 is the least profitable (~21,150)** despite being the most defensive. It loses on both counts: charging an information margin repels uninformed volume (188k vs 200k), and because it displays a flat 0.50 it never prices the direction, so the regime-aware arbitrageur extracts the most from it (+3,455).
- The **guarded** (defence-in-depth) model is solid but slightly behind the plain direction models: the extra information margin prevents arbitrage that was already small, at the cost of a little volume. Over-defending costs more than it saves.
- The original `momentum_lookup` family is mid-pack: full volume but more leakage, because its 1D curve is the easiest to out-predict.

**Takeaway.** For profit, the winner is the model that prices the direction well while keeping
competitive odds (full volume), not the one that defends hardest against arbitrage. Pricing the
direction beats hiding it and over-charging.

**Caveat.** The ranking depends on the demand elasticity (assumed 8.0), the volume scale, and the
arbitrageurs' size. The qualitative result — direction-pricing competitive models win, the
hide-and-charge model 3 trails — is the robust message.

![net house PnL by model](assets/10_pricing_for_profit.png)